# Task 2.3 — Silver: Student-Course Engagement Summary
## Notebook: 06_silver_student_engagement

**Pipeline:** EduPath Learning Analytics · Medallion Architecture
**Layer:** Silver (multi-source aggregate)

**What this notebook does:**
- Aggregates all engagement signals per (student_id, course_id) pair from:
    → bronze.video_watch_bronze       : video_minutes, video_completion_rate
    → bronze.quiz_attempts_bronze     : quiz_pass_rate, quiz_attempts_count
    → silver.discussion_posts_parsed  : discussion_posts_count
    → bronze.lms_events_bronze        : login_count_7d, login_count_30d, assignment_submissions
    → bronze.enrollments_bronze       : last_accessed_date, current_progress_pct
- Computes days_since_last_active from last_accessed_date vs today
- Computes engagement_trend by comparing current week vs prior week login activity
  (IMPROVING / STABLE / DECLINING)
- Writes to silver.student_course_engagement — master input for Task 2.4 dropout risk

**Source tables:**
    mini_project_grp6.bronze.video_watch_bronze
    mini_project_grp6.bronze.quiz_attempts_bronze
    mini_project_grp6.silver.discussion_posts_parsed
    mini_project_grp6.bronze.lms_events_bronze
    mini_project_grp6.bronze.enrollments_bronze

**Target table:** mini_project_grp6.silver.student_course_engagement

**Business Rules enforced:**
- student_id and course_id must not be NULL in output (< 0.5% threshold)
- days_since_last_active must be >= 0
- video_completion_rate must be between 0.0 and 1.0
- quiz_pass_rate must be between 0.0 and 1.0
- engagement_trend must be one of: IMPROVING, STABLE, DECLINING

**Run order:** Run all cells top-to-bottom. Safe to re-run (OVERWRITE mode).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"
SILVER  = "silver"

# --- Source tables ---
VIDEO_BRONZE_TABLE      = f"{CATALOG}.{BRONZE}.video_watch_bronze"
QUIZ_BRONZE_TABLE       = f"{CATALOG}.{BRONZE}.quiz_attempts_bronze"
DISCUSSION_SILVER_TABLE = f"{CATALOG}.{SILVER}.discussion_posts_parsed"
LMS_BRONZE_TABLE        = f"{CATALOG}.{BRONZE}.lms_events_bronze"
ENROLL_BRONZE_TABLE     = f"{CATALOG}.{BRONZE}.enrollments_bronze"

# --- Target table ---
ENGAGEMENT_SILVER_TABLE = f"{CATALOG}.{SILVER}.student_course_engagement"
DQ_AUDIT_TABLE          = f"{CATALOG}.audit.dq_audit"

# --- Engagement trend thresholds ---
# Current week = last 7 days; prior week = 8–14 days ago
# If current_week_logins > prior_week_logins + 1 → IMPROVING
# If current_week_logins < prior_week_logins - 1 → DECLINING
# Otherwise → STABLE
TREND_DELTA_THRESHOLD = 1

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Reference date for days_since_last_active calculation ---
# In production this is current_date(); set here for reproducibility
REFERENCE_DATE = "current_date()"   # will be used as F.current_date() in code

# --- Set default catalog + schema ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(SILVER)

print(f"✅ Catalog  : {CATALOG}")
print(f"✅ Schema   : {SILVER}")
print(f"✅ Target   : {ENGAGEMENT_SILVER_TABLE}")
print()
print("Sources:")
print(f"   {VIDEO_BRONZE_TABLE}")
print(f"   {QUIZ_BRONZE_TABLE}")
print(f"   {DISCUSSION_SILVER_TABLE}")
print(f"   {LMS_BRONZE_TABLE}")
print(f"   {ENROLL_BRONZE_TABLE}")

**Expected output:**
```
✅ Catalog  : mini_project_grp6
✅ Schema   : silver
✅ Target   : mini_project_grp6.silver.student_course_engagement

Sources:
   mini_project_grp6.bronze.video_watch_bronze
   mini_project_grp6.bronze.quiz_attempts_bronze
   mini_project_grp6.silver.discussion_posts_parsed
   mini_project_grp6.bronze.lms_events_bronze
   mini_project_grp6.bronze.enrollments_bronze
```

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    DoubleType, DateType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Video Engagement Metrics

Source: bronze.video_watch_bronze
Grain: one row per (watch_id, student_id, course_id, video_id)

Aggregate per (student_id, course_id):
- total_video_minutes_watched : SUM(watched_seconds) / 60
- video_completion_rate       : AVG(completion_pct) / 100
  (completion_pct is stored as 0–100 in Bronze, we normalise to 0.0–1.0)
- videos_watched_count        : COUNT(DISTINCT video_id)

In [0]:
# ============================================================
# CELL 5 — Video engagement metrics per (student_id, course_id)
#
# total_video_minutes_watched : SUM(watched_seconds) / 60
# video_completion_rate       : AVG(completion_pct) / 100
#                               normalised from 0–100 → 0.0–1.0
# videos_watched_count        : COUNT(DISTINCT video_id)
#
# Only PASSED watch events matter for completion rate.
# We use all rows (including partial watches) for minutes watched.
# ============================================================

video_df = spark.table(VIDEO_BRONZE_TABLE)

video_metrics_df = (
    video_df
    .filter(
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull()
    )
    .groupBy("student_id", "course_id")
    .agg(
        # Total minutes watched — sum of all watched_seconds across videos
        F.round(
            F.sum("watched_seconds") / 60.0, 2
        ).alias("total_video_minutes_watched"),
        # Average completion rate — normalise from 0-100 to 0.0-1.0
        F.round(
            F.avg("completion_pct") / 100.0, 4
        ).alias("video_completion_rate"),
        # How many distinct videos did this student watch in this course
        F.countDistinct("video_id").alias("videos_watched_count"),
    )
)

print("── Video metrics aggregated ──────────────────────")
print(f"   (student_id, course_id) pairs: {video_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
video_metrics_df.select(
    F.min("total_video_minutes_watched").alias("min_video_mins"),
    F.max("total_video_minutes_watched").alias("max_video_mins"),
    F.round(F.avg("total_video_minutes_watched"), 2).alias("avg_video_mins"),
    F.round(F.avg("video_completion_rate"), 4).alias("avg_completion_rate")
).show()

## Part B — Quiz Engagement Metrics

Source: bronze.quiz_attempts_bronze
Grain: one row per attempt (a student can attempt the same quiz multiple times)

Aggregate per (student_id, course_id):
- quiz_pass_rate      : count(status='PASSED') / count(*) across all attempts
- quiz_attempts_count : total number of quiz attempts
- quizzes_passed      : count of distinct quiz_ids where student has at least one PASS

Note: we exclude IN_PROGRESS and TIMED_OUT records from pass rate calculation
since they have no definitive outcome yet.

In [0]:
# ============================================================
# CELL 7 — Quiz engagement metrics per (student_id, course_id)
#
# quiz_pass_rate      : PASSED attempts / total completed attempts
#                       excludes IN_PROGRESS (no outcome yet)
# quiz_attempts_count : total rows per student-course (all statuses)
# quizzes_passed      : distinct quiz_ids with at least one PASSED attempt
# ============================================================

quiz_df = spark.table(QUIZ_BRONZE_TABLE)

# Total attempts per student-course (all statuses)
quiz_attempts_df = (
    quiz_df
    .filter(
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull()
    )
    .groupBy("student_id", "course_id")
    .agg(
        F.count("*").alias("quiz_attempts_count"),
    )
)

# Pass rate: only count completed attempts (PASSED or FAILED)
# IN_PROGRESS and TIMED_OUT excluded from denominator
quiz_pass_df = (
    quiz_df
    .filter(
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull() &
        F.col("status").isin("PASSED", "FAILED")   # completed attempts only
    )
    .groupBy("student_id", "course_id")
    .agg(
        # Pass rate = PASSED / (PASSED + FAILED)
        F.round(
            F.sum(F.when(F.col("status") == "PASSED", 1).otherwise(0)) /
            F.count("*"),
            4
        ).alias("quiz_pass_rate"),
        # Count of distinct quiz_ids with at least one PASS
        F.countDistinct(
            F.when(F.col("status") == "PASSED", F.col("quiz_id"))
        ).alias("quizzes_passed"),
    )
)

# Join attempts count + pass rate
quiz_metrics_df = (
    quiz_attempts_df
    .join(quiz_pass_df, on=["student_id", "course_id"], how="left")
    # Students with only IN_PROGRESS/TIMED_OUT will have NULL pass_rate → fill 0.0
    .fillna({"quiz_pass_rate": 0.0, "quizzes_passed": 0})
)

print("── Quiz metrics aggregated ───────────────────────")
print(f"   (student_id, course_id) pairs: {quiz_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
quiz_metrics_df.select(
    F.min("quiz_pass_rate").alias("min_pass_rate"),
    F.max("quiz_pass_rate").alias("max_pass_rate"),
    F.round(F.avg("quiz_pass_rate"), 4).alias("avg_pass_rate"),
    F.round(F.avg("quiz_attempts_count"), 2).alias("avg_attempts")
).show()

## Part C — Discussion Engagement Metrics

Source: silver.discussion_posts_parsed (Task 2.1 output)
Grain: one row per post

Aggregate per (student_id, course_id):
- discussion_posts_count : total posts made by this student in this course
                           (student posts only — exclude instructor posts)
- threads_participated   : COUNT(DISTINCT thread_id) they posted in

We use the Silver table (not Bronze) because it has the parsed
author_student_id correctly typed and the is_instructor_post flag.

In [0]:
# ============================================================
# CELL 9 — Discussion engagement metrics per (student_id, course_id)
#
# discussion_posts_count : total student posts in this course
#                          is_instructor_post = false only
# threads_participated   : distinct threads the student posted in
# ============================================================

discussion_df = spark.table(DISCUSSION_SILVER_TABLE)

discussion_metrics_df = (
    discussion_df
    .filter(
        F.col("author_student_id").isNotNull() &
        F.col("course_id").isNotNull() &
        (F.col("is_instructor_post") == False)    # student posts only
    )
    .groupBy(
        F.col("author_student_id").alias("student_id"),
        "course_id"
    )
    .agg(
        F.count("post_id").alias("discussion_posts_count"),
        F.countDistinct("thread_id").alias("threads_participated"),
    )
)

print("── Discussion metrics aggregated ─────────────────")
print(f"   (student_id, course_id) pairs: {discussion_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
discussion_metrics_df.select(
    F.min("discussion_posts_count").alias("min_posts"),
    F.max("discussion_posts_count").alias("max_posts"),
    F.round(F.avg("discussion_posts_count"), 2).alias("avg_posts")
).show()

## Part D — Login and Assignment Metrics from LMS Events

Source: bronze.lms_events_bronze
Grain: one row per event

Aggregate per (student_id, course_id):
- login_count_7d          : logins in the last 7 days (event_type = 'login')
- login_count_30d         : logins in the last 30 days
- assignment_submissions  : count of assignment_submit events

For "last 7 days" and "last 30 days" we compare event_date against
the most recent event_date in the entire LMS table (max_event_date).
This is more robust than using current_date() on sample data
where all dates are historical (Jan–Jun 2024).

In [0]:
# ============================================================
# CELL 11 — Login count and assignment submission metrics
#
# Reference date approach for sample data:
#   max_event_date = latest event_date in the Bronze table
#   This ensures "last 7 days" and "last 30 days" windows
#   are always relative to the most recent data, not today's date.
#   In production, replace max_event_date with F.current_date().
#
# login_count_7d   : logins where event_date >= (max_event_date - 7 days)
# login_count_30d  : logins where event_date >= (max_event_date - 30 days)
# assignment_submissions: count of event_type = 'assignment_submit'
# ============================================================

lms_df = spark.table(LMS_BRONZE_TABLE)

# Find the most recent event date in the Bronze table (reference point)
max_event_date = lms_df.agg(F.max("event_date")).collect()[0][0]
print(f"ℹ  Reference date (max_event_date): {max_event_date}")
print(f"   7-day window  : {max_event_date} minus 7 days")
print(f"   30-day window : {max_event_date} minus 30 days")
print()

# Pre-filter to only login and assignment events (reduce data scanned)
lms_filtered_df = (
    lms_df
    .filter(
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull() &
        F.col("event_type").isin("login", "assignment_submit")
    )
)

lms_metrics_df = (
    lms_filtered_df
    .groupBy("student_id", "course_id")
    .agg(
        # Logins in last 7 days (relative to max_event_date)
        F.sum(
            F.when(
                (F.col("event_type") == "login") &
                (F.col("event_date") >= F.date_sub(F.lit(max_event_date), 7)),
                1
            ).otherwise(0)
        ).alias("login_count_7d"),
        # Logins in last 30 days
        F.sum(
            F.when(
                (F.col("event_type") == "login") &
                (F.col("event_date") >= F.date_sub(F.lit(max_event_date), 30)),
                1
            ).otherwise(0)
        ).alias("login_count_30d"),
        # Assignment submissions
        F.sum(
            F.when(F.col("event_type") == "assignment_submit", 1).otherwise(0)
        ).alias("assignment_submissions"),
    )
)

print("── LMS login + assignment metrics aggregated ─────")
print(f"   (student_id, course_id) pairs: {lms_metrics_df.count():,}")
print()

print("── Stats ─────────────────────────────────────────")
lms_metrics_df.select(
    F.round(F.avg("login_count_7d"), 2).alias("avg_logins_7d"),
    F.round(F.avg("login_count_30d"), 2).alias("avg_logins_30d"),
    F.round(F.avg("assignment_submissions"), 2).alias("avg_assignments")
).show()

## Part E — Enrollment Metrics + days_since_last_active

Source: bronze.enrollments_bronze
Grain: one row per enrollment (one enrollment per student-course pair)

We pull:
- last_active_date         : last_accessed_date from enrollment record
- days_since_last_active   : DATEDIFF(max_event_date, last_accessed_date)
- current_progress_pct     : from enrollment record
- enrollment_status        : ACTIVE / COMPLETED / DROPPED / PAUSED

days_since_last_active is the primary dropout risk signal in Task 2.4.

In [0]:
# ============================================================
# CELL 13 — Enrollment metrics per (student_id, course_id)
#
# last_active_date       : last_accessed_date from enrollments
# days_since_last_active : DATEDIFF(reference_date, last_accessed_date)
# current_progress_pct   : already in enrollments Bronze
# enrollment_status      : ACTIVE / COMPLETED / DROPPED / PAUSED
#
# We use max_event_date (computed in Cell 11) as the reference point
# instead of current_date() so sample data produces meaningful results.
# ============================================================

enroll_df = spark.table(ENROLL_BRONZE_TABLE)

enrollment_metrics_df = (
    enroll_df
    .filter(
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull()
    )
    .select(
        "student_id",
        "course_id",
        F.col("last_accessed_date").alias("last_active_date"),
        F.col("current_progress_pct"),
        F.col("status").alias("enrollment_status"),
        # days_since_last_active = reference_date - last_accessed_date
        # Use F.datediff: positive means last access was in the past (correct)
        F.datediff(
            F.lit(max_event_date),
            F.col("last_accessed_date")
        ).alias("days_since_last_active"),
    )
)

print("── Enrollment metrics aggregated ─────────────────")
print(f"   (student_id, course_id) pairs: {enrollment_metrics_df.count():,}")
print()

print("── days_since_last_active stats ──────────────────")
enrollment_metrics_df.select(
    F.min("days_since_last_active").alias("min_days"),
    F.max("days_since_last_active").alias("max_days"),
    F.round(F.avg("days_since_last_active"), 2).alias("avg_days")
).show()

print("── enrollment_status distribution ────────────────")
enrollment_metrics_df.groupBy("enrollment_status").count().show()

## Part F — Compute engagement_trend (IMPROVING / STABLE / DECLINING)

engagement_trend compares login activity between:
- current week  : event_date in last 7 days  (relative to max_event_date)
- prior week    : event_date in prior 8–14 days

Logic:
  If current_week_logins > prior_week_logins + TREND_DELTA_THRESHOLD → IMPROVING
  If current_week_logins < prior_week_logins - TREND_DELTA_THRESHOLD → DECLINING
  Otherwise → STABLE

This is computed directly from lms_events_bronze login events
and then joined into the final engagement table.

In [0]:
# ============================================================
# CELL 15 — engagement_trend: IMPROVING / STABLE / DECLINING
#
# We count login events in:
#   current_week : last 7 days from max_event_date
#   prior_week   : 8 to 14 days before max_event_date
#
# Then compare the two counts to assign the trend label.
# TREND_DELTA_THRESHOLD = 1 (from Cell 2 config)
# ============================================================

lms_logins_df = (
    spark.table(LMS_BRONZE_TABLE)
    .filter(
        F.col("event_type") == "login" &
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull()
    )
)

# # ✅ CORRECT — wrap the equality check in parentheses
# .filter(
#     (F.col("event_type") == "login") &
#     F.col("student_id").isNotNull() &
#     F.col("course_id").isNotNull()
# )

trend_df = (
    lms_logins_df
    .groupBy("student_id", "course_id")
    .agg(
        # Current week logins: last 7 days
        F.sum(
            F.when(
                F.col("event_date") >= F.date_sub(F.lit(max_event_date), 7),
                1
            ).otherwise(0)
        ).alias("current_week_logins"),
        # Prior week logins: 8 to 14 days ago
        F.sum(
            F.when(
                (F.col("event_date") >= F.date_sub(F.lit(max_event_date), 14)) &
                (F.col("event_date") < F.date_sub(F.lit(max_event_date), 7)),
                1
            ).otherwise(0)
        ).alias("prior_week_logins"),
    )
    # Compute trend label
    .withColumn(
        "engagement_trend",
        F.when(
            F.col("current_week_logins") > F.col("prior_week_logins") + TREND_DELTA_THRESHOLD,
            F.lit("IMPROVING")
        ).when(
            F.col("current_week_logins") < F.col("prior_week_logins") - TREND_DELTA_THRESHOLD,
            F.lit("DECLINING")
        ).otherwise(F.lit("STABLE"))
    )
    .select("student_id", "course_id", "engagement_trend",
            "current_week_logins", "prior_week_logins")
)

print("── engagement_trend distribution ─────────────────")
trend_df.groupBy("engagement_trend").count().orderBy("engagement_trend").show()

print("── Sample rows ───────────────────────────────────")
trend_df.show(5, truncate=False)

## Part G — Join All Metrics into Master Engagement Table

Join all five metric DataFrames on (student_id, course_id):
1. enrollment_metrics_df  → base table (all enrolled students)
2. video_metrics_df       → LEFT JOIN (not every student watches video)
3. quiz_metrics_df        → LEFT JOIN (not every student takes quizzes)
4. discussion_metrics_df  → LEFT JOIN (not every student posts)
5. lms_metrics_df         → LEFT JOIN (login counts + assignment submissions)
6. trend_df               → LEFT JOIN (engagement trend)

Using enrollment as the base ensures we keep all enrolled students
even if they have zero activity on any individual signal.
NULL values from LEFT JOINs are filled with 0 (no activity = 0, not unknown).

In [0]:
# ============================================================
# CELL 17 — LEFT JOIN all metric DataFrames on (student_id, course_id)
#
# Base: enrollment_metrics_df (all enrolled student-course pairs)
# All others: LEFT JOIN so students with no activity are kept
#
# After join: fill NULLs from missing activity with 0
# (a student with no video events has 0 minutes watched, not NULL)
# ============================================================

engagement_df = (
    enrollment_metrics_df
    # 1. Join video metrics
    .join(video_metrics_df, on=["student_id", "course_id"], how="left")
    # 2. Join quiz metrics
    .join(quiz_metrics_df, on=["student_id", "course_id"], how="left")
    # 3. Join discussion metrics
    .join(discussion_metrics_df, on=["student_id", "course_id"], how="left")
    # 4. Join LMS login + assignment metrics
    .join(lms_metrics_df, on=["student_id", "course_id"], how="left")
    # 5. Join engagement trend
    .join(
        trend_df.select("student_id", "course_id", "engagement_trend"),
        on=["student_id", "course_id"],
        how="left"
    )
    # Fill NULLs from LEFT JOINs with 0 for numeric activity columns
    .fillna({
        "total_video_minutes_watched": 0.0,
        "video_completion_rate":       0.0,
        "videos_watched_count":        0,
        "quiz_pass_rate":              0.0,
        "quiz_attempts_count":         0,
        "quizzes_passed":              0,
        "discussion_posts_count":      0,
        "threads_participated":        0,
        "login_count_7d":              0,
        "login_count_30d":             0,
        "assignment_submissions":      0,
    })
    # Students with no login data in trend window → STABLE (not enough signal)
    .fillna({"engagement_trend": "STABLE"})
)

print("── Joined engagement DataFrame ───────────────────")
print(f"   Rows (student-course pairs): {engagement_df.count():,}")
print(f"   Columns: {len(engagement_df.columns)}")

In [0]:
# ============================================================
# CELL 18 — Assemble final Silver DataFrame
#           Select and order all output columns cleanly
#           Add Silver metadata columns
# ============================================================

student_course_engagement_df = (
    engagement_df
    .select(
        # ── Identity ──────────────────────────────────────────
        "student_id",
        "course_id",
        "enrollment_status",
        # ── Video signals ─────────────────────────────────────
        "total_video_minutes_watched",
        "video_completion_rate",
        "videos_watched_count",
        # ── Quiz signals ──────────────────────────────────────
        "quiz_pass_rate",
        "quiz_attempts_count",
        "quizzes_passed",
        # ── Discussion signals ────────────────────────────────
        "discussion_posts_count",
        "threads_participated",
        # ── Login + assignment signals ────────────────────────
        "login_count_7d",
        "login_count_30d",
        "assignment_submissions",
        # ── Recency signals ───────────────────────────────────
        "last_active_date",
        "days_since_last_active",
        "current_progress_pct",
        # ── Trend signal ──────────────────────────────────────
        "engagement_trend",
    )
    # Add Silver metadata
    .withColumn("_silver_load_ts", F.current_timestamp())
    .withColumn("_schema_version",  F.lit(SCHEMA_VERSION))
)

print("── Final engagement DataFrame ────────────────────")
print(f"   Rows    : {student_course_engagement_df.count():,}")
print(f"   Columns : {len(student_course_engagement_df.columns)}")
print()
student_course_engagement_df.printSchema()

In [0]:
# ============================================================
# CELL 19 — Write to silver.student_course_engagement
#           Mode: OVERWRITE — fully recomputed from all sources
#           overwriteSchema=true allows schema evolution
# ============================================================

(
    student_course_engagement_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ENGAGEMENT_SILVER_TABLE)
)

final_count = spark.table(ENGAGEMENT_SILVER_TABLE).count()
print(f"✅ Written to: {ENGAGEMENT_SILVER_TABLE}")
print(f"   Rows in table: {final_count:,}")

**Expected output:**
```
✅ Written to: mini_project_grp6.silver.student_course_engagement
   Rows in table: 2,500
```

In [0]:
# ============================================================
# CELL 20 — Verify silver.student_course_engagement
# ============================================================

eng_df = spark.table(ENGAGEMENT_SILVER_TABLE)

print("── student_course_engagement ─────────────────────")
print(f"Total rows   : {eng_df.count():,}")
print(f"Columns      : {len(eng_df.columns)}")
print()

print("── engagement_trend distribution ─────────────────")
eng_df.groupBy("engagement_trend").count().orderBy("engagement_trend").show()

print("── enrollment_status distribution ────────────────")
eng_df.groupBy("enrollment_status").count().orderBy("enrollment_status").show()

print("── key metric stats ──────────────────────────────")
eng_df.select(
    F.round(F.avg("total_video_minutes_watched"), 2).alias("avg_video_mins"),
    F.round(F.avg("video_completion_rate"), 4).alias("avg_video_completion"),
    F.round(F.avg("quiz_pass_rate"), 4).alias("avg_quiz_pass_rate"),
    F.round(F.avg("discussion_posts_count"), 2).alias("avg_discussion_posts"),
    F.round(F.avg("days_since_last_active"), 2).alias("avg_days_inactive"),
    F.round(F.avg("current_progress_pct"), 2).alias("avg_progress_pct")
).show()

print("── sample rows ───────────────────────────────────")
eng_df.select(
    "student_id", "course_id",
    "total_video_minutes_watched", "quiz_pass_rate",
    "discussion_posts_count", "login_count_7d",
    "days_since_last_active", "engagement_trend"
).show(5, truncate=False)

## Part H — Data Quality Checks (Silver)

DQ checks for student_course_engagement:
1. student_id and course_id must not be NULL (< 0.5% threshold)
2. video_completion_rate must be between 0.0 and 1.0
3. quiz_pass_rate must be between 0.0 and 1.0
4. days_since_last_active must be >= 0
5. engagement_trend must be one of IMPROVING / STABLE / DECLINING

All failures written to audit.dq_audit.

In [0]:
# ============================================================
# CELL 22 — DQ Check 1: NULL student_id and course_id
#           These are the primary keys of this table
#           Any NULL here would break Task 2.4 dropout risk scoring
# ============================================================

eng_df = spark.table(ENGAGEMENT_SILVER_TABLE)
total  = eng_df.count()

for col_name in ["student_id", "course_id"]:
    null_count = eng_df.filter(F.col(col_name).isNull()).count()
    pct        = (null_count / total * 100) if total > 0 else 0
    status     = "✅" if null_count == 0 else "⚠"
    alert      = "  ← ALERT: exceeds 0.5% completeness threshold!" if pct > 0.5 else ""

    print(f"── DQ Check 1: NULL {col_name} ─────────────────")
    print(f"   {status}  nulls={null_count:,}  ({pct:.3f}%){alert}")

    if null_count > 0:
        flagged = (
            eng_df.filter(F.col(col_name).isNull())
            .withColumn("dq_check_name", F.lit(f"null_{col_name}_engagement"))
            .withColumn("dq_table",      F.lit(ENGAGEMENT_SILVER_TABLE))
            .withColumn("dq_severity",   F.lit("HIGH"))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.lit(f"{col_name} is NULL in student_course_engagement"))
        )
        (
            flagged
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "student_id", "course_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {null_count} rows flagged → written to {DQ_AUDIT_TABLE}")
    print()

In [0]:
# ============================================================
# CELL 23 — DQ Check 2: video_completion_rate and quiz_pass_rate
#           must be between 0.0 and 1.0
#           Values outside this range indicate aggregation errors
# ============================================================

eng_df = spark.table(ENGAGEMENT_SILVER_TABLE)

for col_name in ["video_completion_rate", "quiz_pass_rate"]:
    invalid_rows = (
        eng_df
        .filter(
            (F.col(col_name) < 0.0) |
            (F.col(col_name) > 1.0)
        )
        .withColumn("dq_check_name", F.lit(f"{col_name}_out_of_range"))
        .withColumn("dq_table",      F.lit(ENGAGEMENT_SILVER_TABLE))
        .withColumn("dq_severity",   F.lit("HIGH"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.concat_ws(" | ",
            F.lit(f"{col_name} out of [0,1]:"),
            F.col(col_name).cast("string"),
            F.lit("student_id:"), F.col("student_id")
        ))
    )
    invalid_count = invalid_rows.count()

    print(f"── DQ Check 2: {col_name} range ───────────────")
    if invalid_count > 0:
        (
            invalid_rows
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "student_id", "course_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {invalid_count} rows flagged → written to {DQ_AUDIT_TABLE}")
        invalid_rows.select("student_id", "course_id", col_name).show(5, truncate=False)
    else:
        print(f"   ✅ PASSED — all {col_name} values in [0.0, 1.0]")
    print()

In [0]:
# ============================================================
# CELL 24 — DQ Check 3a: days_since_last_active must be >= 0
#           A negative value means last_accessed_date is in the future
#           which is a data entry error in the enrollments source
#
#           DQ Check 3b: engagement_trend must be one of the 3 valid values
# ============================================================

eng_df = spark.table(ENGAGEMENT_SILVER_TABLE)

# ── 3a: days_since_last_active ─────────────────────────────
negative_days = (
    eng_df
    .filter(F.col("days_since_last_active") < 0)
    .withColumn("dq_check_name", F.lit("negative_days_since_active"))
    .withColumn("dq_table",      F.lit(ENGAGEMENT_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("days_since_last_active is negative:"),
        F.col("days_since_last_active").cast("string"),
        F.lit("student_id:"), F.col("student_id")
    ))
)

neg_count = negative_days.count()
print("── DQ Check 3a: days_since_last_active ───────────")
if neg_count > 0:
    (
        negative_days
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "student_id", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {neg_count} rows with negative days_since_active → {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED — all days_since_last_active >= 0")

print()

# ── 3b: engagement_trend valid values ─────────────────────
valid_trends = ["IMPROVING", "STABLE", "DECLINING"]

invalid_trend = (
    eng_df
    .filter(~F.col("engagement_trend").isin(valid_trends))
    .withColumn("dq_check_name", F.lit("invalid_engagement_trend"))
    .withColumn("dq_table",      F.lit(ENGAGEMENT_SILVER_TABLE))
    .withColumn("dq_severity",   F.lit("MEDIUM"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("engagement_trend has invalid value:"),
        F.col("engagement_trend")
    ))
)

trend_invalid_count = invalid_trend.count()
print("── DQ Check 3b: engagement_trend valid values ────")
if trend_invalid_count > 0:
    (
        invalid_trend
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "student_id", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {trend_invalid_count} rows with invalid trend → {DQ_AUDIT_TABLE}")
    invalid_trend.groupBy("engagement_trend").count().show()
else:
    print("   ✅ PASSED — all engagement_trend values are IMPROVING / STABLE / DECLINING")

In [0]:
%sql
-- ============================================================
-- CELL 25 — SQL verification of silver.student_course_engagement
-- ============================================================

SELECT
    COUNT(*)                                           AS total_rows,
    COUNT(DISTINCT student_id)                         AS unique_students,
    COUNT(DISTINCT course_id)                          AS unique_courses,
    ROUND(AVG(total_video_minutes_watched), 2)         AS avg_video_mins,
    ROUND(AVG(video_completion_rate), 4)               AS avg_video_completion_rate,
    ROUND(AVG(quiz_pass_rate), 4)                      AS avg_quiz_pass_rate,
    ROUND(AVG(discussion_posts_count), 2)              AS avg_discussion_posts,
    ROUND(AVG(login_count_7d), 2)                      AS avg_logins_7d,
    ROUND(AVG(days_since_last_active), 2)              AS avg_days_inactive,
    SUM(CASE WHEN engagement_trend = 'IMPROVING'  THEN 1 ELSE 0 END) AS improving_count,
    SUM(CASE WHEN engagement_trend = 'STABLE'     THEN 1 ELSE 0 END) AS stable_count,
    SUM(CASE WHEN engagement_trend = 'DECLINING'  THEN 1 ELSE 0 END) AS declining_count
FROM mini_project_grp6.silver.student_course_engagement;

In [0]:
%sql
-- ============================================================
-- CELL 26 — Preview: students who will be HIGH dropout risk in Task 2.4
--           HIGH risk rule: days_since_last_active > 14 AND
--                           current_progress_pct < 30
-- This is a preview only — official risk tiers computed in Task 2.4
-- ============================================================

SELECT
    student_id,
    course_id,
    days_since_last_active,
    current_progress_pct,
    quiz_pass_rate,
    video_completion_rate,
    engagement_trend,
    enrollment_status
FROM mini_project_grp6.silver.student_course_engagement
WHERE days_since_last_active > 14
  AND current_progress_pct < 30
  AND enrollment_status = 'ACTIVE'
ORDER BY days_since_last_active DESC
LIMIT 20;

In [0]:
%sql
# ============================================================
# CELL 27 — Task 2.3 completion summary
# ============================================================

final_df    = spark.table(ENGAGEMENT_SILVER_TABLE)
final_count = final_df.count()
students    = final_df.select("student_id").distinct().count()
courses     = final_df.select("course_id").distinct().count()

improving = final_df.filter(F.col("engagement_trend") == "IMPROVING").count()
stable    = final_df.filter(F.col("engagement_trend") == "STABLE").count()
declining = final_df.filter(F.col("engagement_trend") == "DECLINING").count()

print("═" * 60)
print("  TASK 2.3 COMPLETE — Silver Student-Course Engagement")
print("═" * 60)
print()
print(f"  ✅ {ENGAGEMENT_SILVER_TABLE}")
print(f"      Rows (student-course pairs): {final_count:,}")
print(f"      Unique students             : {students:,}")
print(f"      Unique courses              : {courses:,}")
print(f"      Mode                        : OVERWRITE (fully recomputed)")
print()
print("  engagement_trend breakdown:")
print(f"      IMPROVING  : {improving:,}")
print(f"      STABLE     : {stable:,}")
print(f"      DECLINING  : {declining:,}")
print()
print("  Sources joined:")
print("      bronze.video_watch_bronze      → video_mins, completion_rate")
print("      bronze.quiz_attempts_bronze    → pass_rate, attempts_count")
print("      silver.discussion_posts_parsed → posts_count, threads_participated")
print("      bronze.lms_events_bronze       → login_count_7d/30d, assignments")
print("      bronze.enrollments_bronze      → last_active, progress_pct, status")
print()
print("  DQ checks run:")
print("      Cell 22 — NULL student_id / course_id")
print("      Cell 23 — video_completion_rate + quiz_pass_rate in [0,1]")
print("      Cell 24 — days_since_last_active >= 0")
print("      Cell 24 — engagement_trend valid values")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 2.4 → 07_silver_dropout_risk")
print("         HIGH / MEDIUM / LOW risk tiers from this table")
print("═" * 60)